<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab5-Lenses-Refraction-and-Cooling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 5: Lenses, Refraction and Cooling

**Unit 5: Light and heat** · General Practical Physics I · Davidson Open University

**In this lab you will:**
- find the focal length of a converging lens from object and image distances;
- find a refractive index from Snell's law and a graph;
- calculate heat transferred with Q = mcΔT;
- analyse a cooling curve with Newton's law of cooling.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '5.1': {'h': 'c55d7d96de87631f', 'hint': '1/f = 1/u + 1/v, so f = 1 / (1/u + 1/v).', 'dp': 1},
    '5.2': {'h': '219483c9c41548f1', 'hint': 'm = v / u.', 'ok': 'The image is twice the size of the object and inverted.', 'dp': 2},
    '5.3': {'h': '3da5233f864dbf28', 'hint': 'n₂ = sin θ₁ / sin θ₂ when n₁ = 1. Convert degrees with np.radians.', 'ok': 'Typical of glass or acrylic.', 'dp': 2},
    '5.4': {'h': '2dbb278253608678', 'hint': 'np.polyfit(x, y, 1)[0] with x = sin θ₂ and y = sin θ₁, both from np.radians.', 'dp': 2},
    '5.5': {'h': 'd78e0a2d4c61ca3f', 'hint': 'Q = m * c * ΔT.', 'dp': 0},
    '5.6': {'h': 'e6b4de3a9a33da19', 'hint': 'gradient = np.polyfit(t, np.log(T - Ta), 1)[0]; k = −gradient.', 'ok': 'A larger k means faster cooling.', 'dp': 3},
    '5.7': {'h': '688e3c179ab9bdc1', 'hint': 'Make a list of f values with zip(u_values, v_values), then return the mean rounded to 1 dp.', 'ok': 'Averaging several pairs reduces random error.', 'dp': 1, 'args': [([30, 40, 60], [60, 40, 30]), ([25, 50], [100/3, 50/3*2]), ([15, 20], [30, 20])]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Geometric Optics](https://phet.colorado.edu/en/simulations/geometric-optics)** – move an object in front of a converging lens and measure the image distance.
- **[PhET Bending Light](https://phet.colorado.edu/en/simulations/bending-light)** – measure angles of incidence and refraction with the protractor.
- **[Newton's Law of Cooling virtual lab (Virtual Labs, Amrita)](https://ht-amrt.vlabs.ac.in/exp/newtons-law-cooling/)** – record temperature against time for a cooling liquid. Free.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Lens formula** (real-is-positive): 1/f = 1/u + 1/v, magnification = v/u. **Snell's law:** n₁ sin θ₁ = n₂ sin θ₂. **Heat:** Q = mcΔT. **Newton's law of cooling:** T − Tₐ = (T₀ − Tₐ) e^(−kt), so ln(T − Tₐ) against t is a straight line with gradient −k.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
u, v = 25.0, 100/3        # cm
print('f =', round(1 / (1/u + 1/v), 2), 'cm')

## Part B – Practise (about 20 minutes)
**B1.** An object 30 cm from a converging lens forms a real image 60 cm from the lens. Find f in cm.

In [ ]:
answer_5_1 = ...

In [ ]:
check('5.1', answer_5_1)

In [ ]:
answer_5_2 = ...   # the magnification v/u for the same lens

In [ ]:
check('5.2', answer_5_2)

**B3.** Light passes from air (n₁ = 1.00) into a block. The angle of incidence is 40° and the angle of refraction is 25.4°. Find n₂ (2 decimal places). *NumPy uses radians:* `np.sin(np.radians(40))`.

In [ ]:
answer_5_3 = ...

In [ ]:
check('5.3', answer_5_3)

**B4. Refractive index from a graph.** Plot sin θ₁ (y) against sin θ₂ (x) for the illustrative Bending Light data below. The gradient is n₂. Give it to 2 decimal places.

In [ ]:
theta1 = np.array([10, 20, 30, 40, 50, 60])                  # degrees (illustrative)
theta2 = np.array([6.7, 13.2, 19.5, 25.4, 30.7, 35.3])       # degrees
plt.plot(np.sin(np.radians(theta2)), np.sin(np.radians(theta1)), 'o'); plt.xlabel('sin θ₂'); plt.ylabel('sin θ₁'); plt.show()

In [ ]:
answer_5_4 = ...

In [ ]:
check('5.4', answer_5_4)

**B5. Heat.** How many joules warm 0.20 kg of water (c = 4200 J/kg°C) by 15 °C?

In [ ]:
answer_5_5 = ...

In [ ]:
check('5.5', answer_5_5)

**B6. Cooling curve.** Water cools in a room at Tₐ = 28 °C (illustrative data). Fit ln(T − Tₐ) against t and give the cooling constant k (per minute) as a positive number, 3 decimal places.

In [ ]:
t = np.array([0, 5, 10, 15, 20, 25, 30])                      # minutes (illustrative)
T = np.array([80.0, 68.5, 59.5, 52.6, 47.1, 42.9, 39.6])      # °C
Ta = 28.0
plt.plot(t, T, 'o-'); plt.xlabel('time (min)'); plt.ylabel('temperature (°C)'); plt.show()

In [ ]:
answer_5_6 = ...

In [ ]:
check('5.6', answer_5_6)

## Part C – Build a tool (about 20 minutes)
Complete `focal_length(u_values, v_values)`. For several object/image pairs it should compute f = 1/(1/u + 1/v) for each pair and return the **mean** f, rounded to 1 decimal place.

In [ ]:
def focal_length(u_values, v_values):
    ...      # YOUR CODE

In [ ]:
check('5.7', focal_length)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 5 assignment.** In PhET Geometric Optics, record four object/image distance pairs and find f with your tool. In Bending Light, take five angle pairs and find n from a graph. Label the data "simulated". *Safety:* do not heat water at home for this lab; use the online cooling lab instead.

In [ ]:
my_u = [...]
my_v = [...]

**D2. Reflect.**
1. Why is a graph of sin θ₁ against sin θ₂ better than one pair of angles?
2. What would happen to k if the room were colder?
3. Why must simulated data be labelled in your report?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 5 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.